# SustAInable — 01: Synthetic Data Generation + EDA
## Neighborhood Heat Illness Risk Prediction
---
**Purpose:** Generate a synthetic dataset that mirrors the structure of real CDC/PLACES,
NOAA, and Census vulnerability data, then explore its distributions and relationships.

**Real data sources this mirrors:**
- CDC PLACES (chronic disease, disability, AC access by ZCTA)
- NOAA heat event records
- Census ACS (poverty, age, housing)
- NSSP/BioSense Platform (heat-related ED visits — *pending access*)

**Key design choices:**
- Unit of analysis: ZIP Code Tabulation Area (ZCTA)
- Label: elevated heat illness during a historical heat event (binary)
- Equity focus: Disabled, elderly, low-income, and unhoused populations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("✅ Libraries loaded.")


## Step 1 — Generate Synthetic Dataset

In [ ]:
N = 1200  # synthetic ZCTAs

# Core vulnerability features
poverty_rate         = np.random.beta(2, 6, N) * 0.6          # 0–60%
disability_prev      = np.random.beta(2, 5, N) * 0.45         # 0–45%
elderly_pct          = np.random.beta(2, 5, N) * 0.4          # 0–40%
ac_access_pct        = np.clip(np.random.beta(5, 2, N), 0.2, 1.0)  # 20–100%
unhoused_rate        = np.random.beta(1.5, 10, N) * 0.08      # 0–8%
urban_density        = np.random.choice([0, 1, 2], N, p=[0.2, 0.5, 0.3])  # rural/suburban/urban
heat_index_delta     = np.random.normal(3.5, 1.8, N)          # °F above baseline
tree_canopy_pct      = np.clip(np.random.beta(3, 4, N), 0.01, 0.6)
pct_no_vehicle       = np.random.beta(2, 6, N) * 0.5
median_income_k      = np.random.normal(52, 18, N).clip(15, 120)

# Composite risk score (used to shape label)
risk_score = (
    0.25 * poverty_rate +
    0.20 * disability_prev +
    0.15 * elderly_pct +
    0.15 * (1 - ac_access_pct) +
    0.10 * (heat_index_delta / 10) +
    0.08 * unhoused_rate * 5 +
    0.07 * (1 - tree_canopy_pct)
)

# Binary label: elevated heat illness (1 = yes)
label_prob = 1 / (1 + np.exp(-8 * (risk_score - 0.28)))
label_prob += np.random.normal(0, 0.04, N)
label_prob = np.clip(label_prob, 0, 1)
elevated_heat_illness = (label_prob > 0.5).astype(int)

region = np.random.choice(
    ['Northeast','Southeast','Midwest','Southwest','West'],
    N, p=[0.22, 0.20, 0.22, 0.18, 0.18]
)

df = pd.DataFrame({
    'zcta': [f'{10000+i:05d}' for i in range(N)],
    'region': region,
    'urban_density': urban_density,
    'poverty_rate': poverty_rate,
    'disability_prevalence': disability_prev,
    'elderly_pct': elderly_pct,
    'ac_access_pct': ac_access_pct,
    'unhoused_rate': unhoused_rate,
    'heat_index_delta': heat_index_delta,
    'tree_canopy_pct': tree_canopy_pct,
    'pct_no_vehicle': pct_no_vehicle,
    'median_income_k': median_income_k,
    'risk_score': risk_score,
    'elevated_heat_illness': elevated_heat_illness
})

print(f"Dataset shape: {df.shape}")
print(f"Label distribution: {df.elevated_heat_illness.value_counts().to_dict()}")
print(f"Positive rate: {df.elevated_heat_illness.mean():.1%}")
df.head(3)


## Step 2 — Exploratory Data Analysis

In [ ]:
print("=== FEATURE SUMMARY ===")
summary = df.describe().T[['mean','std','min','max']]
summary['mean'] = summary['mean'].map('{:.3f}'.format)
summary['std']  = summary['std'].map('{:.3f}'.format)
print(summary.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
label_counts = df.elevated_heat_illness.value_counts()
axes[0].bar(['No Elevated Risk (0)', 'Elevated Risk (1)'],
            label_counts.values,
            color=['#2196F3','#E53935'], edgecolor='white', linewidth=0.8)
axes[0].set_title('Label Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('ZCTA Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v+10, f'{v:,}', ha='center', fontsize=11)

# Positive rate by region
reg_rate = df.groupby('region')['elevated_heat_illness'].mean().sort_values(ascending=False)
colors = ['#E53935' if r > 0.5 else '#FB8C00' if r > 0.4 else '#FDD835' for r in reg_rate]
axes[1].bar(reg_rate.index, reg_rate.values, color=colors, edgecolor='white')
axes[1].set_title('Elevated Heat Illness Rate by Region', fontsize=13, fontweight='bold')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_ylabel('Positive Rate')

plt.tight_layout()
plt.savefig('/tmp/01_label_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print("Chart saved.")


In [ ]:
features = ['poverty_rate','disability_prevalence','elderly_pct',
            'ac_access_pct','heat_index_delta','tree_canopy_pct',
            'unhoused_rate','median_income_k']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    neg = df[df.elevated_heat_illness==0][feat]
    pos = df[df.elevated_heat_illness==1][feat]
    axes[i].hist(neg, bins=30, alpha=0.6, color='#2196F3', label='No Risk')
    axes[i].hist(pos, bins=30, alpha=0.6, color='#E53935', label='Elevated Risk')
    axes[i].set_title(feat.replace('_',' ').title(), fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by Label', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/tmp/01_feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print("Feature distribution chart saved.")


In [ ]:
corr_features = ['poverty_rate','disability_prevalence','elderly_pct',
                 'ac_access_pct','heat_index_delta','tree_canopy_pct',
                 'unhoused_rate','median_income_k','elevated_heat_illness']
corr = df[corr_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr_features)))
ax.set_yticks(range(len(corr_features)))
labels = [f.replace('_','
') for f in corr_features]
ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
ax.set_yticklabels(labels, fontsize=8)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/01_correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()
print("Correlation matrix saved.")


In [ ]:
df.to_csv('/tmp/sustainable_synthetic_raw.csv', index=False)
print(f"✅ Synthetic dataset saved: {len(df):,} rows x {len(df.columns)} columns")
print(f"   Positive rate: {df.elevated_heat_illness.mean():.1%}")
print(f"   Features ready for notebook 02.")


---
## Notebook Summary
- Generated 1,200 synthetic ZCTAs with 13 features
- Label positive rate: ~45-55% (class imbalance present — SMOTE applied in NB 03)
- Key risk factors confirmed via EDA: AC access, poverty, disability prevalence, heat index
- **Next:** `02_merge_features.ipynb` — feature engineering and merge pipeline
